# 1kg_eur — sharded GRM on Batch

Computes the genomic relatedness matrix (GRM) via `plink --make-grm-bin
--parallel k N_SHARDS`, distributed across Google Batch tasks using `dsub`.

Each task localizes the full BED panel once and runs its assigned shard range
sequentially, writing `.grm.bin` shards to the bucket.

**Prerequisites:**
- `04_grm_panel_qc.ipynb` — BED + .frq staged to `03_grm/grm_input/`
- plink 1.9 binary staged to the bucket (Cell: stage plink binary)

**Dimensions:** ~1.69M variants × 223,209 individuals.
Full GRM: N(N+1)/2 × 4 bytes ≈ 99.6 GB across all shards.

## Config

In [ ]:
import math, os, subprocess

PROJECT_ID      = "wb-swift-sprout-7231"
REGION          = "us-central1"
SERVICE_ACCOUNT = "pet-27799165194323faf22e2@wb-swift-sprout-7231.iam.gserviceaccount.com"
NETWORK         = f"projects/{PROJECT_ID}/global/networks/network"
SUBNETWORK      = f"projects/{PROJECT_ID}/regions/{REGION}/subnetworks/subnetwork"
CLOUD_SDK_TAG   = "581.0.0-slim"

WS_GS  = "gs://cloned-shared-env-pilot-wb-swift-sprout-7231/phenotypic_covariance_v9"
R_GS   = f"{WS_GS}/1kg_eur"

GRM_INPUT_GS = f"{R_GS}/03_grm/grm_input"
SHARD_OUT_GS = f"{R_GS}/03_grm/shards"
PLINK_BIN_GS = f"{R_GS}/03_grm/bin/plink"
LOG_GS       = f"{R_GS}/03_grm/logs"

BED_NAME = "1kg_CEUGBR_GRM_QC"

# ── machine sizing ─────────────────────────────────────────────────────────────
# BED is ~94 GB; plink holds two copies in memory during --make-grm-bin.
# Memory = (BED_SIZE * 2 + 4 GB overhead), rounded up to 256 MB boundary.
# N1 custom needs memory in exact multiples of 256 MB and -ext suffix if > 8192 MB/vCPU.
BED_SIZE_GB  = 94          # verify with: gcloud storage ls -l $GRM_INPUT_GS/$BED_NAME.bed
MACHINE_VCPUS = 16
_raw_mb = int((BED_SIZE_GB * 2 + 4) * 1024)
MEMORY_MB = math.ceil(_raw_mb / 256) * 256
_mem_per_vcpu = MEMORY_MB // MACHINE_VCPUS
MACHINE_TYPE = (f"n1-custom-{MACHINE_VCPUS}-{MEMORY_MB}-ext"
                if _mem_per_vcpu > 8192
                else f"n1-custom-{MACHINE_VCPUS}-{MEMORY_MB}")
PLINK_MEM_MB = MEMORY_MB - 8192
DISK_SIZE_GB = 200   # 94 GB panel + shard outputs

# ── sharding ───────────────────────────────────────────────────────────────────
# N_SHARDS × N_TASKS must cover all work; each task runs N_SHARDS/N_TASKS shards.
# 1kg_eur (n=223K) is ~2× harder per shard than eur_D2 (n=155K) because GRM scales as N².
N_SHARDS = 20   # --parallel k N_SHARDS
N_TASKS  = 5    # dsub tasks; each gets N_SHARDS/N_TASKS = 4 shards

for k, v in dict(MACHINE_TYPE=MACHINE_TYPE, MEMORY_MB=MEMORY_MB,
                 PLINK_MEM_MB=PLINK_MEM_MB, N_SHARDS=N_SHARDS, N_TASKS=N_TASKS,
                 GRM_INPUT_GS=GRM_INPUT_GS, SHARD_OUT_GS=SHARD_OUT_GS).items():
    print(f"  {k}: {v}")

## Install dsub and patch wrapper image

dsub hardcodes a `CLOUD_SDK_IMAGE` tag in `providers/google_utils.py` for five
wrapper runnables (localize / log-stream / delocalize). `--image` only sets the
sixth (user-command) runnable. All tags pinned in any released dsub version have
been withdrawn from gcr.io; patching is mandatory in the same shell as the
submission call — `pip install dsub` silently reverts it.

In [ ]:
%%bash
pip install --quiet --upgrade 'dsub>=0.5.3'
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
echo "--- wrapper image tool (must be 'gcloud storage cp', not 'gsutil') ---"
grep -o 'gcloud storage cp\|gsutil .*cp' "$DSUB_DIR/providers/google_utils.py" | sort -u

## Stage plink 1.9 binary (one-time)

In [ ]:
%%bash
set -eo pipefail
BIN_DIR="$HOME/bin"; mkdir -p "$BIN_DIR"
if [ ! -x "$BIN_DIR/plink" ]; then
  # copy from a previously staged location (VPC-SC blocks S3 downloads)
  gcloud storage cp "${WS_GS}/03_grm_shards/eur_D2/bin/plink" "$BIN_DIR/plink" 2>/dev/null &&     chmod +x "$BIN_DIR/plink" || {
    echo "plink 1.9 not found — upload manually and re-run"
    exit 1
  }
fi
"$BIN_DIR/plink" --version

# stage to bucket for Batch workers
gcloud storage cp "$BIN_DIR/plink" "$PLINK_BIN_GS"
gcloud storage ls -l "$PLINK_BIN_GS" 

## Smoke test

In [ ]:
%%bash
set -eo pipefail
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E "s|cloud-sdk:[0-9]+\.[0-9]+\.[0-9]+-slim|cloud-sdk:${CLOUD_SDK_TAG}|g"   "$DSUB_DIR/providers/google_utils.py"
grep CLOUD_SDK_IMAGE "$DSUB_DIR/providers/google_utils.py"

dsub   --provider google-batch --project "$PROJECT_ID" --regions "$REGION"   --logging "$LOG_GS"   --service-account "$SERVICE_ACCOUNT"   --network "$NETWORK" --subnetwork "$SUBNETWORK" --use-private-address   --image "gcr.io/google.com/cloudsdktool/cloud-sdk:${CLOUD_SDK_TAG}"   --name "eur-r2-smoke"   --machine-type "n1-standard-2" --disk-size 10   --env SHARD_OUT_GS="$SHARD_OUT_GS"   --command '
    set -x
    hostname; date
    echo ok > /tmp/smoke.txt
    gcloud storage cp /tmp/smoke.txt "${SHARD_OUT_GS}/smoke_$(date +%s).txt"       && echo WROTE_OK || echo WRITE_FAILED
  ' 2>&1 | tee /tmp/smoke_job.log
echo "smoke job: $(tail -1 /tmp/smoke_job.log)" | tee /tmp/smoke_job_id.txt

In [ ]:
%%bash
dstat --provider google-batch --project "$PROJECT_ID"   --location "$REGION" --jobs "$(cat /tmp/smoke_job_id.txt)"   --users '*' --status '*' --full 2>&1 | tail -20

## Preflight check

In [ ]:
%%bash
echo "=== panel ==="
for ext in bed bim fam; do
  gcloud storage ls -l "${GRM_INPUT_GS}/${BED_NAME}.$ext" 2>/dev/null     || echo "  MISSING: ${BED_NAME}.$ext"
done

echo "=== frequencies ==="
gcloud storage ls -l "${GRM_INPUT_GS}/${BED_NAME}_freq.frq" 2>/dev/null   || echo "  MISSING: ${BED_NAME}_freq.frq — run 04_grm_panel_qc.ipynb"

echo "=== plink binary ==="
gcloud storage ls -l "$PLINK_BIN_GS" 2>/dev/null || echo "  not staged"

echo "=== sample count ==="
gcloud storage cat "${GRM_INPUT_GS}/${BED_NAME}.fam" 2>/dev/null | wc -l   || echo "  (could not read .fam)"

echo "=== BED size (update BED_SIZE_GB if different from 94) ==="
gcloud storage ls -l "${GRM_INPUT_GS}/${BED_NAME}.bed"   | awk '{printf "  %.1f GB\n", $1/1024/1024/1024}' 2>/dev/null || true

## Submit shard jobs

Each task receives a contiguous range of shards. Task k runs shards
`(k-1)*(N_SHARDS/N_TASKS)+1` through `k*(N_SHARDS/N_TASKS)`.

`plink --make-grm-bin --parallel k N` writes shard k of an N-shard GRM.
The row-range recovery used in accumulation is calibrated to plink 1.9's
`--parallel` split algorithm — do not use plink2 here.

In [ ]:
# build tasks TSV: TASK_SHARDS column lists shard numbers as "k1,k2,..."
shards_per_task = N_SHARDS // N_TASKS
tasks_tsv = "/tmp/1kg_eur_grm_tasks.tsv"
with open(tasks_tsv, "w") as fh:
    fh.write("--env TASK_SHARDS\n")
    for t in range(N_TASKS):
        shards = list(range(t * shards_per_task + 1,
                             (t + 1) * shards_per_task + 1))
        fh.write(",".join(map(str, shards)) + "\n")
print(f"tasks file written: {tasks_tsv}")
print(open(tasks_tsv).read())

In [ ]:
%%bash
set -eo pipefail
DSUB_DIR=$(python -c "import dsub, os; print(os.path.dirname(dsub.__file__))")
sed -i -E "s|cloud-sdk:[0-9]+\.[0-9]+\.[0-9]+-slim|cloud-sdk:${CLOUD_SDK_TAG}|g"   "$DSUB_DIR/providers/google_utils.py"

dsub   --provider google-batch --project "$PROJECT_ID" --regions "$REGION"   --logging "$LOG_GS"   --service-account "$SERVICE_ACCOUNT"   --network "$NETWORK" --subnetwork "$SUBNETWORK" --use-private-address   --image "gcr.io/google.com/cloudsdktool/cloud-sdk:${CLOUD_SDK_TAG}"   --name "eur-r2-grm"   --machine-type "$MACHINE_TYPE" --disk-size "$DISK_SIZE_GB"   --input  PLINK_BIN="$PLINK_BIN_GS"   --input-recursive BED_DIR="$GRM_INPUT_GS"   --output-recursive SHARD_DIR="$SHARD_OUT_GS"   --env    BED_NAME="$BED_NAME"   --env    N_SHARDS="$N_SHARDS"   --env    PLINK_MEM_MB="$PLINK_MEM_MB"   --tasks /tmp/1kg_eur_grm_tasks.tsv   --command '
    set -eo pipefail
    chmod +x "$PLINK_BIN"
    BED_PREFIX="${BED_DIR}/${BED_NAME}"
    FREQ_PATH="${BED_DIR}/${BED_NAME}_freq.frq"
    IFS="," read -ra SHARDS <<< "$TASK_SHARDS"
    for k in "${SHARDS[@]}"; do
      echo "--- shard $k / $N_SHARDS ---"
      "$PLINK_BIN"         --bfile "$BED_PREFIX"         --read-freq "$FREQ_PATH"         --make-grm-bin --parallel "$k" "$N_SHARDS"         --memory "$PLINK_MEM_MB"         --out "${SHARD_DIR}/grm.shard${k}"
      rm -f "${SHARD_DIR}/grm.shard${k}.grm.N.bin"
    done
  ' 2>&1 | tee /tmp/grm_jobs.log
cat /tmp/grm_jobs.log | tee /tmp/grm_job_id.txt

In [ ]:
%%bash
dstat --provider google-batch --project "$PROJECT_ID"   --location "$REGION" --jobs "$(cat /tmp/grm_job_id.txt)"   --users '*' --status '*' --full 2>&1 | tail -30

## Copy notebook to bucket

In [ ]:
import subprocess, os
_nb = os.path.expanduser('~/repos/AOU-covariance/notebooks/06_grm_shards.ipynb')
_gs = f'{OUT_GS}/notebooks/06_grm_shards.ipynb'
subprocess.run(['gcloud', 'storage', 'cp', _nb, _gs], check=True)
print(f'notebook -> {_gs}')